In [1]:
import yaml
import numpy as np
import zarr

from enum import StrEnum
from tqdm  import tqdm
from typing import Optional
from dataclasses import dataclass, field
from pathlib import Path

In [3]:
from kungfu_zarro import (
    DataKind,
    WriteType,
    ControlVar,
    AxisSpec,
    ReadoutSpec,
    MeasurementSchema,
    StreamWriter
)

## **Testing Zarr**

In [4]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 10, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpec([Vgt]),
        AxisSpec([Vch, Vm])
    ],
    measure = [
        ReadoutSpec(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpec(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpec(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

In [5]:
root_path1 = "C:/Users/User/data/2026-02-24/exp_001"
root_path2 = "/Users/helium/Documents/data/2026-02-24/exp_001"
root_path3 = "/Users/helium/Documents/data/2026-02-24"

writer = StreamWriter(
    root=root_path3,
    experiment_id="exp_005",
    schema=mschema,
    target_file_size_bytes = 1 * 1024 * 1024,  # target ~5MB shards; adjust as needed
    overwrite=True,
)

S21 None (10, 200, 1601)


In [6]:
writer._dataset_map["lockin_X"].info

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200)
Chunk shape        : (10, 200)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 8000 (7.8K)

In [7]:
writer._dataset_map["S21"].info

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Chunk shape        : (10, 200, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)

In [8]:
writer._control_shape

(10, 200)

In [51]:
for i in tqdm(range(len(Vgt.values))):
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()
        writer.write_point((i, j), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

100%|██████████| 10/10 [00:07<00:00,  1.39it/s]


In [42]:
for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    writer.write_trace((i,), {"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

100%|██████████| 10/10 [00:00<00:00, 110.88it/s]


In [10]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_all({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})
writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 10/10 [00:00<00:00, 303.45it/s]


## **Testing 1D sweeps**

In [78]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpec([Vch, Vm])
    ],
    measure = [
        ReadoutSpec(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpec(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpec(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.POINTWISE,
)

root_path1 = "C:/Users/User/data/2026-02-24/exp_001"
root_path2 = "/Users/helium/Documents/data/2026-02-24/exp_001"
root_path3 = "/Users/helium/Documents/data/2026-02-24"

writer = StreamWriter(
    root=root_path3,
    experiment_id="exp_006",
    schema=mschema,
    target_file_size_bytes = 1 * 1024 * 1024,  # target ~5MB shards; adjust as needed
    overwrite=True,
)

S21 (200, 1601) (1, 1601)


### **all**

In [73]:
MAG = []
LOCKIN = []
TEMP = []
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    MAG.append(mag)
    LOCKIN.append(lockinx)
    TEMP.append(T1K)

writer.write_all({"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

### **point-wise**

In [79]:
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    writer.write_point((j,), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

In [80]:
z2 = zarr.open(root_path3 + "/exp_006" + '/vault.zarr', mode='r')

In [81]:
z2.info

Name        : 
Type        : Group
Zarr format : 3
Read-only   : True
Store type  : LocalStore

In [82]:
z2["S21"][:,159]

array([0.71217304, 0.7791478 , 0.25337303, 0.8110178 , 0.62843966,
       0.21710882, 0.44674513, 0.3223809 , 0.9283747 , 0.5560198 ,
       0.15175974, 0.89349633, 0.21961947, 0.27907264, 0.7857904 ,
       0.30240563, 0.36837342, 0.5701076 , 0.14658768, 0.88254684,
       0.5552247 , 0.37003762, 0.58824456, 0.14646049, 0.05471136,
       0.4754062 , 0.18136322, 0.4731433 , 0.49360225, 0.6483784 ,
       0.7570852 , 0.9044214 , 0.69816667, 0.48448277, 0.59736276,
       0.5467565 , 0.30280182, 0.40438667, 0.26610735, 0.6808312 ,
       0.36121178, 0.21524431, 0.72429097, 0.08302695, 0.7513417 ,
       0.67296475, 0.22753534, 0.804547  , 0.6851803 , 0.77642226,
       0.9323398 , 0.32019186, 0.34458354, 0.13228817, 0.9268819 ,
       0.72835934, 0.33093235, 0.6731499 , 0.5255012 , 0.42007732,
       0.40932298, 0.42303807, 0.6059848 , 0.5412287 , 0.0541488 ,
       0.52568465, 0.36453784, 0.26975012, 0.9730776 , 0.96484345,
       0.9645322 , 0.42478496, 0.20967433, 0.7199395 , 0.39358